#### The aim of this project is to build a predictive model that can accurately forecast **sales performance based on weather conditions and historical sales data**. 
#### By integrating weather features (temperature, humidity, rainfall) with sales records, the project seeks to:

In [11]:
# 1) Imports and constants
# Import required libraries for data handling, visualization, and machine learning
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
from pathlib import Path

In [17]:
# Modeling Libraries and tools
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score# Visualization
# Importing library for Saving model
import joblib

In [19]:
#1). Visualization
import matplotlib.pyplot as plt
import seaborn as sns

In [27]:
# Load CSV file
DATA_PATH = Path("sales_weather forecast.csv")
OUTPUT_DIR = Path("model_output")
OUTPUT_DIR.mkdir(exist_ok=True)

In [29]:
df = pd.read_csv(DATA_PATH)
print("Initial shape:", df.shape)
print(df.head())

Initial shape: (65535, 23)
   STORE_ID        STORE_NAME    CITY COUNTRY  PRODUCT_ID  \
0       104  MaxiStore Madrid  Madrid   Spain         695   
1       104  MaxiStore Madrid  Madrid   Spain         695   
2       104  MaxiStore Madrid  Madrid   Spain         695   
3       104  MaxiStore Madrid  Madrid   Spain         695   
4       104  MaxiStore Madrid  Madrid   Spain         695   

          PRODUCT_NAME         CATEGORY SALES_DATE  TOTAL_UNITS_SOLD  \
0  Heavy Coat (Medium)  Winter Clothing    00:00.0                 5   
1  Heavy Coat (Medium)  Winter Clothing    00:00.0                 3   
2  Heavy Coat (Medium)  Winter Clothing    00:00.0                19   
3  Heavy Coat (Medium)  Winter Clothing    00:00.0                18   
4  Heavy Coat (Medium)  Winter Clothing    00:00.0                11   

   TOTAL_REVENUE  ...  MIN_TEMP  AVG_SUNSHINE  AVG_WEATHER_CODE  \
0         135.55  ... -1.162500  30210.970700               0.0   
1          81.33  ...  8.987499   4162.

In [31]:
# 3) Data preprocessing
# Detecting date column
date_col = None
for c in df.columns:
    if "date" in c.lower():
        date_col = c
        break

# Parse and sort by date if exists
if date_col:
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    df = df.dropna(subset=[date_col]).sort_values(date_col)

# Extract date features
if date_col:
    df["year"] = df[date_col].dt.year
    df["month"] = df[date_col].dt.month
    df["day"] = df[date_col].dt.day
    df["weekday"] = df[date_col].dt.weekday

    # Seasonality encoding
    df["month_sin"] = np.sin(2*np.pi*df["month"]/12)
    df["month_cos"] = np.cos(2*np.pi*df["month"]/12)
    df["weekday_sin"] = np.sin(2*np.pi*df["weekday"]/7)
    df["weekday_cos"] = np.cos(2*np.pi*df["weekday"]/7)

In [33]:
# 4) Split features and target
# Use TOTAL_UNITS_SOLD as target 
if "TOTAL_UNITS_SOLD" in df.columns:
    TARGET = "TOTAL_UNITS_SOLD"
elif "TOTAL_REVENUE" in df.columns:
    TARGET = "TOTAL_REVENUE"
else:
    raise ValueError("No suitable target column found.")

X = df.drop(columns=[TARGET])
y = df[TARGET]

numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()

In [35]:
# 5) Training and testing split

if date_col:
    split_idx = int(len(df)*0.8)
    X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
    y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

In [37]:
# 6) Building Model Pipelines
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_cols),
    ("cat", categorical_transformer, cat_cols)
])

In [39]:
# 7) Training the Model
# Lightweight, fast model
dt = DecisionTreeRegressor(random_state=42, max_depth=8)

pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", dt)
])

pipe.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['STORE_ID', 'PRODUCT_ID',
                                                   'TOTAL_REVENUE',
                                                   'AVG_UNIT_PRICE', 'AVG_TEMP',
                                                   'MAX_TEMP', 'MIN_TEMP',
                                                   'AVG_SUNSHINE',
                                                   'AVG_WEATHER_CODE',
                                                   'ROLLING_AVG_7D_UNITS',
                                                   'ROLLING_AVG_30D_UNITS',
                                                   'ROLLING_SUM_7D_...
                                                   'LAG_30D_UNITS', 'year',
                                                   'month', 'day', 'weekday',
                                                   'month_sin', 'month_cos',
                                                   'weekday_sin',
                                                   'weekday_cos']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['STORE_NAME', 'CITY',
                                                   'COUNTRY', 'PRODUCT_NAME',
                                                   'CATEGORY'])])),
                ('model', DecisionTreeRegressor(max_depth=8, random_state=42))])

In [41]:
# 8) Evaluating the Model
y_pred = pipe.predict(X_test)

rmse = mean_squared_error(y_test, y_pred, squared=False)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Evaluation Metrics:")
print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)

Evaluation Metrics:
RMSE: 1.6270907181949423
MAE: 0.7834078785780008
R2: 0.9335668245593811


In [65]:
# Plot actual vs predicted
plt.figure(figsize=(10,5))
plt.plot(y_test.values, label="Actual", alpha=0.7)
plt.plot(y_pred, label="Predicted", alpha=0.7)
plt.legend()
plt.title("Actual vs Predicted Sales (Decision Tree)")
plt.savefig(OUTPUT_DIR/"actual_vs_predicted_dt.png")
plt.show()

In [49]:
 # 9) Saving the model
joblib.dump(pipe, OUTPUT_DIR/"sales_forecast_dt_model.joblib")
print("Model training complete. Outputs saved in:", OUTPUT_DIR)

Model training complete. Outputs saved in: model_output


##### Model Interpretation

##### What this means for the forecasting sales with weather

##### Conclusion

#### By 

## Ebenezer Adebiyi

### Linkedin : Ebenezer Adebiyi
### Email : ebenezerdadebiyi@gmail.com